In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(
    "../data/processed/merged_dataset.csv",
    parse_dates=["day"]
)

print(df.shape)
df.head()

(3469352, 45)


,LCLid,day,energy_median,energy_mean,energy_max,energy_count,energy_std,energy_sum,energy_min,stdorToU,...,temperatureHigh,sunriseTime,temperatureHighTime,uvIndexTime,summary,temperatureLowTime,apparentTemperatureMin,apparentTemperatureMaxTime,apparentTemperatureLowTime,moonPhase
0,MAC000131,2011-12-16,0.1415,0.296167,1.116,48,0.281471,14.216,0.031,Std,...,4.53,2011-12-16 08:01:35,2011-12-16 15:00:00,2011-12-16 11:00:00,Mostly cloudy throughout the day.,2011-12-17 08:00:00,-2.65,2011-12-16 00:00:00,2011-12-17 08:00:00,0.70
1,MAC000131,2011-12-17,0.1015,0.189812,0.685,48,0.188405,9.111,0.064,Std,...,5.35,2011-12-17 08:02:21,2011-12-17 14:00:00,2011-12-17 11:00:00,Partly cloudy throughout the day.,2011-12-18 07:00:00,-3.56,2011-12-17 15:00:00,2011-12-18 06:00:00,0.73
2,MAC000131,2011-12-18,0.1140,0.218979,0.676,48,0.202919,10.511,0.065,Std,...,5.49,2011-12-18 08:03:04,2011-12-18 14:00:00,2011-12-18 12:00:00,Partly cloudy until evening.,2011-12-19 01:00:00,-4.12,2011-12-18 14:00:00,2011-12-19 02:00:00,0.77
3,MAC000131,2011-12-19,0.1910,0.325979,0.788,48,0.259205,15.647,0.066,Std,...,6.64,2011-12-19 08:03:43,2011-12-19 19:00:00,2011-12-19 11:00:00,Partly cloudy throughout the day.,2011-12-20 04:00:00,-3.67,2011-12-19 19:00:00,2011-12-20 08:00:00,0.81
4,MAC000131,2011-12-20,0.2180,0.357500,1.077,48,0.287597,17.160,0.066,Std,...,8.26,2011-12-20 08:04:20,2011-12-20 12:00:00,2011-12-20 11:00:00,Partly cloudy throughout the day.,2011-12-21 00:00:00,1.68,2011-12-20 12:00:00,2011-12-20 23:00:00,0.85


In [4]:
df["year"] = df["day"].dt.year

df["month"] = df["day"].dt.month

df["day_of_month"] = df["day"].dt.day

df["day_of_week"] = df["day"].dt.dayofweek

df["week"] = df["day"].dt.isocalendar().week.astype(int)

df["quarter"] = df["day"].dt.quarter

In [5]:
df["is_weekend"] = df["day_of_week"].isin([5,6]).astype(int)

In [6]:
def get_season(month):
    if month in [12,1,2]:
        return "Winter"
    elif month in [3,4,5]:
        return "Spring"
    elif month in [6,7,8]:
        return "Summer"
    else:
        return "Autumn"

df["season"] = df["month"].apply(get_season)

In [7]:
df[
    [
        "day",
        "year",
        "month",
        "day_of_month",
        "day_of_week",
        "week",
        "quarter",
        "is_weekend",
        "season"
    ]
].head()

,day,year,month,day_of_month,day_of_week,week,quarter,is_weekend,season
0,2011-12-16,2011,12,16,4,50,4,0,Winter
1,2011-12-17,2011,12,17,5,50,4,1,Winter
2,2011-12-18,2011,12,18,6,50,4,1,Winter
3,2011-12-19,2011,12,19,0,51,4,0,Winter
4,2011-12-20,2011,12,20,1,51,4,0,Winter


In [8]:
df = df.sort_values(["LCLid", "day"]).reset_index(drop=True)

In [9]:
df["lag_1"] = (
    df.groupby("LCLid")["energy_sum"]
      .shift(1)
)

In [10]:
df["lag_7"] = (
    df.groupby("LCLid")["energy_sum"]
      .shift(7)
)

In [11]:
df["lag_30"] = (
    df.groupby("LCLid")["energy_sum"]
      .shift(30)
)

In [12]:
df[
    [
        "LCLid",
        "day",
        "energy_sum",
        "lag_1",
        "lag_7",
        "lag_30"
    ]
].head(40)

,LCLid,day,energy_sum,lag_1,lag_7,lag_30
0,MAC000002,2012-10-13,11.087,NaN,NaN,NaN
1,MAC000002,2012-10-14,13.223,11.087,NaN,NaN
2,MAC000002,2012-10-15,10.257,13.223,NaN,NaN
3,MAC000002,2012-10-16,9.769,10.257,NaN,NaN
4,MAC000002,2012-10-17,10.885,9.769,NaN,NaN
5,MAC000002,2012-10-18,10.751,10.885,NaN,NaN
6,MAC000002,2012-10-19,8.431,10.751,NaN,NaN
7,MAC000002,2012-10-20,17.378,8.431,11.087,NaN
8,MAC000002,2012-10-21,24.490,17.378,13.223,NaN
9,MAC000002,2012-10-22,18.885,24.490,10.257,NaN


In [13]:
df["rolling_mean_7"] = (
    df.groupby("LCLid")["energy_sum"]
      .transform(lambda x: x.shift(1).rolling(7).mean())
)

In [14]:
df["rolling_std_7"] = (
    df.groupby("LCLid")["energy_sum"]
      .transform(lambda x: x.shift(1).rolling(7).std())
)

In [15]:
df[
    [
        "LCLid",
        "day",
        "energy_sum",
        "lag_1",
        "lag_7",
        "rolling_mean_7",
        "rolling_std_7"
    ]
].head(20)

,LCLid,day,energy_sum,lag_1,lag_7,rolling_mean_7,rolling_std_7
0,MAC000002,2012-10-13,11.087,NaN,NaN,NaN,NaN
1,MAC000002,2012-10-14,13.223,11.087,NaN,NaN,NaN
2,MAC000002,2012-10-15,10.257,13.223,NaN,NaN,NaN
3,MAC000002,2012-10-16,9.769,10.257,NaN,NaN,NaN
4,MAC000002,2012-10-17,10.885,9.769,NaN,NaN,NaN
5,MAC000002,2012-10-18,10.751,10.885,NaN,NaN,NaN
6,MAC000002,2012-10-19,8.431,10.751,NaN,NaN,NaN
7,MAC000002,2012-10-20,17.378,8.431,11.087,10.629000,1.456492
8,MAC000002,2012-10-21,24.490,17.378,13.223,11.527714,2.955606
9,MAC000002,2012-10-22,18.885,24.490,10.257,13.137286,5.765204


In [16]:
df = df.dropna().reset_index(drop=True)

print(df.shape)

(3302637, 58)


In [17]:
df = df.drop(
    columns=[
        "temperatureMaxTime",
        "temperatureMinTime",
        "apparentTemperatureMinTime",
        "apparentTemperatureHighTime",
        "temperatureHighTime",
        "uvIndexTime",
        "temperatureLowTime",
        "apparentTemperatureMaxTime",
        "apparentTemperatureLowTime",
        "sunriseTime",
        "sunsetTime",
        "time",
        "file"
    ]
)

In [18]:
print(df.shape)

df.head()

(3302637, 45)


,LCLid,day,energy_median,energy_mean,energy_max,energy_count,energy_std,energy_sum,energy_min,stdorToU,...,day_of_week,week,quarter,is_weekend,season,lag_1,lag_7,lag_30,rolling_mean_7,rolling_std_7
0,MAC000002,2012-11-15,0.154,0.204333,0.962,48,0.155064,9.808,0.075,Std,...,3,46,4,0,Autumn,10.820,11.663,11.087,11.735571,1.201988
1,MAC000002,2012-11-16,0.217,0.273250,1.341,48,0.231304,13.116,0.074,Std,...,4,46,4,0,Autumn,9.808,13.137,13.223,11.470571,1.407559
2,MAC000002,2012-11-17,0.162,0.203542,0.806,48,0.143391,9.770,0.075,Std,...,5,46,4,1,Autumn,13.116,13.245,10.257,11.467571,1.403432
3,MAC000002,2012-11-18,0.231,0.287062,1.207,48,0.233282,13.779,0.075,Std,...,6,46,4,1,Autumn,9.770,10.699,9.769,10.971143,1.279005
4,MAC000002,2012-11-19,0.136,0.211396,0.796,48,0.159598,10.147,0.075,Std,...,0,47,4,0,Autumn,13.779,12.321,10.885,11.411143,1.646710


In [19]:
df.to_csv(
    "../data/processed/feature_dataset.csv",
    index=False
)

print("Feature dataset saved successfully!")

Feature dataset saved successfully!
